In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, warnings, hashlib
warnings.filterwarnings('ignore')

import numpy as np
from pairwise_aggregation import validate_replicates
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from plot_violin import (
    compute_rbo_distance, compute_jaccard_distance, compute_l2_distance,
    compute_rbo_baseline, compute_jaccard_baseline, compute_l2_baseline,
)

DATASETS   = ['german', 'diabetes', 'acs', 'gmsc']
MODELS     = ['dt', 'rf', 'xgb', 'mlp', 'ftt']
METHODS    = ['random', 'kmeans', 'pred_stratified', 'cte']
METRICS    = ['l2', 'jac', 'rbo']

DATASET_LABELS = {'german': 'German', 'diabetes': 'Diabetes', 'acs': 'ACS', 'gmsc': 'GMSC'}
MODEL_LABELS   = {'dt': 'DT', 'rf': 'RF', 'xgb': 'XGB', 'mlp': 'MLP', 'ftt': 'FTT'}
METHOD_LABELS  = {'random': 'Random', 'kmeans': 'KMeans',
                  'pred_stratified': 'Pred-Strat', 'cte': 'CTE'}
METRIC_LABELS  = {'l2': 'L2 Distance', 'jac': 'Top-3 Jaccard Distance', 'rbo': '1-RBO'}

N_BG_SEEDS = 5
JAC_K      = 3
N_CHUNKS   = {'german': 1, 'diabetes': 1, 'acs': 100, 'gmsc': 100}
N_FEATS    = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}
N_CHUNKS_OVERRIDE = {('gmsc', 'ftt'): 300}  # gmsc/ftt rerun with 300 chunks (2h limit)
RES_DIR    = 'results_tree'
TRUE_DIR   = 'results_true'  # Full-training empirical-distribution reference.
# New B figures are separate from the historical tables/figures_true exports.
TABLE_DIR  = os.path.join('figures_b', 'background_true', 'tables')

def get_n_chunks(ds, model):
    return N_CHUNKS_OVERRIDE.get((ds, model), N_CHUNKS[ds])

print('Constants loaded.')

In [ ]:
# ── Complete empirical-reference groups; no silent row/NaN removal ──────────
_file_cache, _file_rows, _file_features = {}, {}, {}
_reference_cache = {}
reference_status = []


def _load_pkl(path):
    if path not in _file_cache:
        if not os.path.exists(path):
            _file_cache[path] = None
        else:
            with open(path,'rb') as handle:
                obj = pickle.load(handle)
            _file_cache[path] = np.asarray(obj.values)
            _file_rows[path] = getattr(obj,'data',None)
            names = getattr(obj,'feature_names',None)
            _file_features[path] = tuple(names) if names is not None else None
    return _file_cache[path]


def _true_path(ds,model,c=None):
    suffix = f'_c{c}' if c is not None else ''
    return os.path.join(TRUE_DIR,f'{ds}_{model}_0_0_fulltrain{suffix}_sv.pkl')


def _bg_path(ds,model,method,s_idx,c=None):
    suffix = f'_c{c}' if c is not None else ''
    return os.path.join(RES_DIR,f'{ds}_{model}_0_0_{method}_s{s_idx}{suffix}_sv.pkl')


def load_true_chunk(ds, model, c=None):
    return _load_pkl(_true_path(ds,model,c))


def load_bg_chunk(ds, model, method, s_idx, c=None):
    return _load_pkl(_bg_path(ds,model,method,s_idx,c))


def dist_per_instance(metric, sv_m, sv_t):
    if metric == 'l2':  return compute_l2_distance(sv_m, sv_t)
    if metric == 'jac': return compute_jaccard_distance(sv_m, sv_t, k=JAC_K)
    if metric == 'rbo': return compute_rbo_distance(sv_m, sv_t)


def _load_complete(paths):
    parts = [_load_pkl(path) for path in paths]
    if any(part is None for part in parts):
        raise ValueError(f'Missing {sum(part is None for part in parts)}/{len(paths)} required files.')
    values = np.concatenate(parts,axis=0)
    validate_replicates([values],expected_runs=1)
    data = [_file_rows.get(path) for path in paths]
    names = [_file_features.get(path) for path in paths]
    known_names = [name for name in names if name is not None]
    if known_names and any(name != known_names[0] for name in known_names):
        raise ValueError('Feature order differs between reference/input chunks.')
    row_hash = None
    if all(rows is not None for rows in data):
        rows = np.concatenate(data,axis=0)
        if len(rows) != len(values):
            raise ValueError('Stored query data and attribution row counts differ.')
        hashed = pd.util.hash_pandas_object(pd.DataFrame(rows),index=False)
        row_hash = hashlib.sha256(hashed.to_numpy().tobytes()).hexdigest()
    return values,row_hash,(known_names[0] if known_names else None)


def compute_distances(ds, model, method, s_idx):
    """Return one run's N raw distances only from a complete 5-run group.

    Same historical paths/chunk counts, including fulltrain300 vs sampled100
    for GMSC/FTT. Missing/invalid inputs are skipped, not truncated or imputed.
    """
    key = (ds,model,method)
    if key not in _reference_cache:
        record = dict(dataset=ds,model=model,method=method,
                      reference_input_root=TRUE_DIR,sampled_input_root=RES_DIR)
        try:
            n_true, n_bg = get_n_chunks(ds,model), N_CHUNKS[ds]
            true_chunks = [None] if n_true==1 else range(n_true)
            bg_chunks = [None] if n_bg==1 else range(n_bg)
            reference,row_ref,features_ref = _load_complete([
                _true_path(ds,model,c) for c in true_chunks])
            runs = [_load_complete([_bg_path(ds,model,method,run,c)
                                    for c in bg_chunks])
                    for run in range(N_BG_SEEDS)]
            arrays = validate_replicates([run[0] for run in runs])
            if any(array.shape != reference.shape for array in arrays):
                raise ValueError('Reference/run shapes differ; refusing to truncate.')
            known_rows = [row for row in [row_ref]+[run[1] for run in runs]
                          if row is not None]
            known_names = [name for name in [features_ref]+[run[2] for run in runs]
                           if name is not None]
            if known_rows and any(row != known_rows[0] for row in known_rows):
                raise ValueError('Stored reference/run query representations differ; encoding correspondence requires verification.')
            if known_names and any(name != known_names[0] for name in known_names):
                raise ValueError('Stored reference/run feature order differs.')
            matrices = {mt:np.stack([dist_per_instance(mt,array,reference)
                                    for array in arrays]) for mt in METRICS}
            if any(matrix.shape != (N_BG_SEEDS,len(reference))
                   or not np.isfinite(matrix).all() for matrix in matrices.values()):
                raise ValueError('Invalid reference-distance shape or non-finite value.')
            _reference_cache[key] = matrices
            record.update(status='computed',n_runs=N_BG_SEEDS,
                          n_instances=len(reference),n_comparisons=N_BG_SEEDS*len(reference),
                          reference_chunks=n_true,sampled_chunks_per_run=n_bg,
                          row_identity='verified' if row_ref and all(run[1] for run in runs)
                          else 'unverified')
            print(f'{ds}/{model}/{method}: {N_BG_SEEDS} runs x {len(reference)} instances '
                  f'= {N_BG_SEEDS*len(reference)} reference comparisons')
        except ValueError as error:
            _reference_cache[key] = None
            record.update(status='skipped',reason=str(error),
                          n_runs=0,n_instances=0,n_comparisons=0)
            print(f'[SKIP reference] {ds}/{model}/{method}: {error}')
        reference_status.append(record)
    matrices = _reference_cache[key]
    return None if matrices is None else {mt:matrix[s_idx] for mt,matrix in matrices.items()}


In [ ]:
# ── Main computation loop ─────────────────────────────────────────────────────
# table_mean[metric][ds][model][method] = mean of per-instance distances (avg over seeds)
# table_std [metric][ds][model][method] = mean of per-seed instance SDs.
# It is neither an SE nor the SD of instance-level mean distances.
# This reference uses the empirical training distribution, not exact True SHAP.

table_mean = {mt: {ds: {m: {} for m in MODELS} for ds in DATASETS} for mt in METRICS}
table_std  = {mt: {ds: {m: {} for m in MODELS} for ds in DATASETS} for mt in METRICS}

for ds in DATASETS:
    for model in MODELS:
        # Check for a full-training-background reference for this (ds, model).
        n_ch = get_n_chunks(ds, model)
        has_true = load_true_chunk(ds, model, c=None if n_ch==1 else 0) is not None
        if not has_true:
            for mt in METRICS:
                for method in METHODS:
                    table_mean[mt][ds][model][method] = float('nan')
                    table_std [mt][ds][model][method] = float('nan')
            print(f'{ds}/{model}: no full-training-background reference, skip')
            continue

        for method in METHODS:
            seed_means = {mt: [] for mt in METRICS}
            seed_stds  = {mt: [] for mt in METRICS}

            for s_idx in range(N_BG_SEEDS):
                result = compute_distances(ds, model, method, s_idx)
                if result is None:
                    continue
                for mt in METRICS:
                    d = result[mt]
                    if len(d) > 0:
                        seed_means[mt].append(np.mean(d))
                        seed_stds [mt].append(np.std(d))

            for mt in METRICS:
                if seed_means[mt]:
                    table_mean[mt][ds][model][method] = float(np.mean(seed_means[mt]))
                    table_std [mt][ds][model][method] = float(np.mean(seed_stds [mt]))
                else:
                    table_mean[mt][ds][model][method] = float('nan')
                    table_std [mt][ds][model][method] = float('nan')

        print(f'{ds}/{model}: done')

print('All done.')

In [ ]:
# ── Heatmaps: one figure per (metric, stat) ──────────────────────────────────
import matplotlib
try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    pass

matplotlib.rcParams.update({
    'font.size':        13,
    'axes.labelsize':   14,
    'xtick.labelsize':  12,
    'ytick.labelsize':  12,
})

os.makedirs(TABLE_DIR, exist_ok=True)

# Omit entirely unverified datasets from displayed rows; retain every
# skipped group in reference_status.csv rather than drawing blank result cells.
PLOT_DATASETS = [ds for ds in DATASETS
                 if any(np.isfinite(table_mean[mt][ds][model].get(method, np.nan))
                        for mt in METRICS for model in MODELS for method in METHODS)]
ROW_LABELS = [f'{DATASET_LABELS[ds]} / {MODEL_LABELS[m]}'
              for ds in PLOT_DATASETS for m in MODELS]
COL_LABELS = [METHOD_LABELS[m] for m in METHODS]
ROW_BOUNDS = [k * len(MODELS) for k in range(1, len(PLOT_DATASETS))]

def make_annot(mat, fmt):
    """Custom annotation: empty string for NaN, formatted number otherwise."""
    out = np.empty(mat.shape, dtype=object)
    for idx in np.ndindex(mat.shape):
        v = mat[idx]
        out[idx] = '' if np.isnan(v) else format(v, fmt)
    return out

for mt in METRICS:
    for stat_label, tbl in [('Mean', table_mean), ('Std', table_std)]:
        if not PLOT_DATASETS:
            continue
        fig, ax = plt.subplots(figsize=(6, max(3, 2 * len(PLOT_DATASETS))))

        mat = np.full((len(PLOT_DATASETS) * len(MODELS), len(METHODS)), float('nan'))
        for i, ds in enumerate(PLOT_DATASETS):
            for j, model in enumerate(MODELS):
                for l, method in enumerate(METHODS):
                    mat[i * len(MODELS) + j, l] = tbl[mt][ds][model][method]

        df_heat = pd.DataFrame(mat, index=ROW_LABELS, columns=COL_LABELS)
        vmax = float(np.nanpercentile(mat, 95)) if not np.all(np.isnan(mat)) else 1.0

        sns.heatmap(df_heat, ax=ax, vmin=0, vmax=vmax, cmap='YlOrRd',
                    annot=make_annot(mat, '.3f'), fmt='', annot_kws={'size': 10},
                    linewidths=0.3, linecolor='white',
                    cbar_kws={'label': stat_label, 'shrink': 0.6})

        for rb in ROW_BOUNDS:
            ax.axhline(rb, color='black', linewidth=1.5)

        ax.set_xlabel('Sampling Method', fontsize=14)
        ax.tick_params(axis='x', rotation=30, labelsize=12)
        ax.tick_params(axis='y', rotation=0,  labelsize=12)

        fig.tight_layout()
        fname = f'{TABLE_DIR}/true_heatmap_{mt}_{stat_label.lower()}.pdf'
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

In [ ]:
# ── GMSC-only heatmaps (for paper) ────────────────────────────────────────────
DS_GMSC = 'gmsc'
gmsc_row_labels = [f'{DATASET_LABELS[DS_GMSC]} / {MODEL_LABELS[m]}' for m in MODELS]

for mt in METRICS:
    for stat_label, tbl in [('Mean', table_mean), ('Std', table_std)]:
        mat_gmsc = np.full((len(MODELS), len(METHODS)), float('nan'))
        for j, model in enumerate(MODELS):
            for l, method in enumerate(METHODS):
                mat_gmsc[j, l] = tbl[mt][DS_GMSC][model][method]

        df_gmsc = pd.DataFrame(mat_gmsc, index=gmsc_row_labels, columns=COL_LABELS)
        vmax = float(np.nanpercentile(mat_gmsc, 95)) if not np.all(np.isnan(mat_gmsc)) else 1.0

        fig, ax = plt.subplots(figsize=(6, 3))
        sns.heatmap(df_gmsc, ax=ax, vmin=0, vmax=vmax, cmap='YlOrRd',
                    annot=make_annot(mat_gmsc, '.3f'), fmt='', annot_kws={'size': 10},
                    linewidths=0.3, linecolor='white',
                    cbar_kws={'label': stat_label, 'shrink': 0.6})

        ax.set_xlabel('Sampling Method', fontsize=14)
        ax.tick_params(axis='x', rotation=30, labelsize=12)
        ax.tick_params(axis='y', rotation=0, labelsize=12)

        fig.tight_layout()
        fname = f'{TABLE_DIR}/true_heatmap_{mt}_{stat_label.lower()}_gmsc.pdf'
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

In [ ]:
# ── % above baseline heatmap (l2 + jac + rbo) ────────────────────────────────

BL_METRICS = ['l2', 'jac', 'rbo']

bl_thresholds = {}
for ds in DATASETS:
    d = N_FEATS[ds]
    r = max(1, d // 3)
    bl_thresholds[ds] = {
        'l2':  compute_l2_baseline(d=d, r=r)['min'],
        'jac': compute_jaccard_baseline(d=d, r=JAC_K)['min'],
        'rbo': compute_rbo_baseline(n_features=d)['min'],
    }
    print(f"{ds}: l2>{bl_thresholds[ds]['l2']:.4f}  "
          f"jac>{bl_thresholds[ds]['jac']:.4f}  "
          f"rbo>{bl_thresholds[ds]['rbo']:.4f}")

# Percentage of reference comparisons above baseline: pool all 5 x N
# raw reference distances, then threshold. This is not sensitive instances/N.
pct_data = {mt: {} for mt in BL_METRICS}
for mt in BL_METRICS:
    for ds in DATASETS:
        pct_data[mt][ds] = {}
        thr = bl_thresholds[ds][mt]
        for model in MODELS:
            pct_data[mt][ds][model] = {}
            for method in METHODS:
                all_dists = []
                for s_idx in range(N_BG_SEEDS):
                    result = compute_distances(ds, model, method, s_idx)
                    if result is None:
                        continue
                    d_arr = result[mt]
                    valid = d_arr  # Complete finite 5-run group validated by compute_distances.
                    if len(valid): all_dists.append(valid)
                if all_dists:
                    flat = np.concatenate(all_dists)
                    pct_data[mt][ds][model][method] = 100.0 * np.mean(flat > thr)
                else:
                    pct_data[mt][ds][model][method] = float('nan')
        print(f'  computed: {ds}/{mt}')

# One heatmap per metric
for mt in BL_METRICS:
    if not PLOT_DATASETS:
        continue
    fig, ax = plt.subplots(figsize=(6, max(3, 2 * len(PLOT_DATASETS))))

    mat = np.full((len(PLOT_DATASETS) * len(MODELS), len(METHODS)), float('nan'))
    for i, ds in enumerate(PLOT_DATASETS):
        for j, model in enumerate(MODELS):
            for l, method in enumerate(METHODS):
                mat[i * len(MODELS) + j, l] = pct_data[mt][ds][model][method]

    df_heat = pd.DataFrame(mat, index=ROW_LABELS, columns=COL_LABELS)
    sns.heatmap(df_heat, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                annot=make_annot(mat, '.0f'), fmt='', annot_kws={'size': 10},
                linewidths=0.3, linecolor='white',
                cbar_kws={'label': '% reference comparisons > baseline', 'shrink': 0.6})

    for rb in ROW_BOUNDS:
        ax.axhline(rb, color='black', linewidth=1.5)

    ax.set_xlabel('Sampling Method', fontsize=14)
    ax.tick_params(axis='x', rotation=30, labelsize=12)
    ax.tick_params(axis='y', rotation=0,  labelsize=12)

    fig.tight_layout()
    fname = f'{TABLE_DIR}/true_pct_baseline_{mt}.pdf'
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# ── GMSC-only % above baseline heatmaps (for paper) ──────────────────────────
DS_GMSC = 'gmsc'
gmsc_row_labels = [f'{DATASET_LABELS[DS_GMSC]} / {MODEL_LABELS[m]}' for m in MODELS]

for mt in BL_METRICS:
    mat_gmsc = np.full((len(MODELS), len(METHODS)), float('nan'))
    for j, model in enumerate(MODELS):
        for l, method in enumerate(METHODS):
            mat_gmsc[j, l] = pct_data[mt][DS_GMSC][model][method]

    df_gmsc = pd.DataFrame(mat_gmsc, index=gmsc_row_labels, columns=COL_LABELS)

    fig, ax = plt.subplots(figsize=(6, 3))
    sns.heatmap(df_gmsc, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                annot=make_annot(mat_gmsc, '.0f'), fmt='', annot_kws={'size': 10},
                linewidths=0.3, linecolor='white',
                cbar_kws={'label': '% reference comparisons > baseline', 'shrink': 0.6})

    ax.set_xlabel('Sampling Method', fontsize=14)
    ax.tick_params(axis='x', rotation=30, labelsize=12)
    ax.tick_params(axis='y', rotation=0,  labelsize=12)

    fig.tight_layout()
    fname = f'{TABLE_DIR}/true_pct_baseline_{mt}_gmsc.pdf'
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# ── Violin plot helpers ───────────────────────────────────────────────────────
import matplotlib.patches as mpatches

JACCARD_VALUES = np.array([0.0, 0.5, 0.8, 1.0])
JACCARD_HATCHES = ['', '..', '//', 'xx']
JACCARD_ALPHAS = [0.30, 0.50, 0.75, 1.0]


def jaccard_proportions(values):
    """Four exact raw Top-3 distances; reject averaged or unexpected values."""
    arr = np.asarray(values, dtype=float).reshape(-1)
    if not arr.size:
        return None
    if not np.isfinite(arr).all():
        raise ValueError('Jaccard frequencies require finite pair distances.')
    matches = np.isclose(arr[:, None], JACCARD_VALUES[None, :],
                         rtol=0.0, atol=1e-8)
    if not np.all(matches.sum(axis=1) == 1):
        raise ValueError('Expected raw Top-3 Jaccard distances: 0, .5, .8, 1.')
    return matches.mean(axis=0)


def discrete_jaccard_bars(ax, data_list, positions, colors, widths=0.35):
    """Stack category frequencies; preserve original group colors/positions."""
    artists = []
    for values, pos, color in zip(data_list, positions, colors):
        fractions = jaccard_proportions(values)
        if fractions is None:
            continue
        bottom = 0.0
        for fraction, hatch, alpha in zip(
                fractions, JACCARD_HATCHES, JACCARD_ALPHAS):
            height = 100.0 * fraction
            artists.extend(ax.bar(pos, height, bottom=bottom, width=widths,
                                  color=color, alpha=alpha, edgecolor='black',
                                  linewidth=0.35, hatch=hatch))
            bottom += height
    handles = [mpatches.Patch(facecolor='0.55', alpha=alpha,
                              edgecolor='black', hatch=hatch,
                              label=f'{distance:g}')
               for distance, hatch, alpha in zip(
                   JACCARD_VALUES, JACCARD_HATCHES, JACCARD_ALPHAS)]
    ax.legend(handles=handles, title='Top-3 Jaccard distance', ncol=4,
              loc='lower center', bbox_to_anchor=(0.5, 1.02),
              fontsize=9, title_fontsize=9, handlelength=1.3)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Reference comparisons (%)')
    return artists


try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    pass

METHOD_COLORS = {
    'random':          '#5B9BD5',
    'kmeans':          '#E87070',
    'pred_stratified': '#52A788',
    'cte':             '#D4824E',
}

FIG_TRUE_DIR = os.path.join('figures_b', 'background_true', 'distributions')
os.makedirs(FIG_TRUE_DIR, exist_ok=True)

METRIC_YLABEL = {
    'l2':  'L2 Distance',
    'jac': 'Top-3 Jaccard Distance',
    'rbo': '1-RBO',
}

def styled_violin(ax, data_list, positions, colors, widths=0.6, alpha=0.75,
                  metric=None):
    if metric == 'jac':
        return discrete_jaccard_bars(ax, data_list, positions, colors, widths)
    if not data_list: return None
    parts = ax.violinplot(data_list, positions=positions, widths=widths,
                          showmedians=True, showextrema=False)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c); pc.set_edgecolor('none'); pc.set_alpha(alpha)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(1.5)
    parts['cmedians'].set_zorder(5)
    return parts

def add_baseline_true(ax, metric, ds):
    d = N_FEATS[ds]
    if metric == 'l2':
        bl = compute_l2_baseline(d=d, r=max(1, d // 3))
    elif metric == 'jac':
        bl = compute_jaccard_baseline(d=d, r=JAC_K)
    elif metric == 'rbo':
        bl = compute_rbo_baseline(n_features=d)
    else:
        return None
    lo, hi = bl['min'], bl['max']
    if metric == 'jac':
        ax.text(0.02, 0.02, f'Baseline threshold: > {lo:.3f}',
                transform=ax.transAxes, fontsize=9, color='darkred',
                bbox=dict(facecolor='white', alpha=0.9, edgecolor='none'))
        return None
    ax.axhspan(lo, hi, alpha=0.15, color='red', zorder=0)
    ax.axhline(lo, linestyle='--', color='red', linewidth=1.2, alpha=0.5, zorder=0)
    ax.axhline(hi, linestyle='--', color='red', linewidth=1.2, alpha=0.5, zorder=0)
    return mpatches.Patch(facecolor='red', alpha=0.3, label='Baseline reference range')

print('Violin helpers ready.')

In [ ]:
# ── Violin plots: one figure per (dataset, model, metric) ────────────────────

for ds in DATASETS:
    for model in MODELS:
        n_ch = get_n_chunks(ds, model)
        c0 = None if n_ch == 1 else 0
        if load_true_chunk(ds, model, c0) is None:
            continue

        violin_data = {mt: [] for mt in METRICS}
        any_valid = False

        for xi, method in enumerate(METHODS):
            all_dists = {mt: [] for mt in METRICS}
            for s_idx in range(N_BG_SEEDS):
                result = compute_distances(ds, model, method, s_idx)
                if result is None: continue
                for mt in METRICS:
                    d = result[mt]
                    if len(d): all_dists[mt].append(d)
            for mt in METRICS:
                arr = np.concatenate(all_dists[mt]) if all_dists[mt] else np.array([])
                violin_data[mt].append((xi, arr))
            if any(len(all_dists[mt]) for mt in METRICS):
                any_valid = True

        if not any_valid:
            continue

        for mt in METRICS:
            nonempty = [(xi, arr) for xi, arr in violin_data[mt] if len(arr)]
            if not nonempty:
                continue

            fig, ax = plt.subplots(figsize=(5, 4))
            styled_violin(ax,
                          [arr for _, arr in nonempty],
                          [xi  for xi, _  in nonempty],
                          [METHOD_COLORS[METHODS[xi]] for xi, _ in nonempty], metric=mt)

            bl_patch = add_baseline_true(ax, mt, ds)

            ax.set_xticks(range(len(METHODS)))
            ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS], rotation=15, fontsize=13)
            ax.set_xlim(-0.6, len(METHODS) - 0.4)
            ax.set_ylabel('Reference comparisons (%)' if mt == 'jac' else METRIC_YLABEL[mt], fontsize=14)
            ax.set_xlabel('Sampling Method', fontsize=14)
            ax.tick_params(axis='x', length=0, labelsize=13)
            ax.tick_params(axis='y', labelsize=13)

            method_patches = [mpatches.Patch(facecolor=METHOD_COLORS[m], alpha=0.75,
                                             label=METHOD_LABELS[m]) for m in METHODS]
            handles = method_patches + ([bl_patch] if bl_patch else [])
            if mt != 'jac':
                ax.legend(handles=handles, fontsize=9, loc='upper right')

            fig.tight_layout()
            fname = os.path.join(FIG_TRUE_DIR, f'true_violin_{ds}_{model}_{mt}.pdf')
            fig.savefig(fname, bbox_inches='tight')
            plt.show(); plt.close(fig)
            print(f'Saved: {fname}')

In [ ]:
# ── Combined violin per (dataset, model): 3 subplots (l2 | jac | rbo) ────────

for ds in DATASETS:
    for model in MODELS:
        n_ch = get_n_chunks(ds, model)
        c0 = None if n_ch == 1 else 0
        if load_true_chunk(ds, model, c0) is None:
            continue

        violin_data = {mt: [] for mt in METRICS}
        any_valid = False

        for xi, method in enumerate(METHODS):
            all_dists = {mt: [] for mt in METRICS}
            for s_idx in range(N_BG_SEEDS):
                result = compute_distances(ds, model, method, s_idx)
                if result is None: continue
                for mt in METRICS:
                    d = result[mt]
                    if len(d): all_dists[mt].append(d)
            for mt in METRICS:
                arr = np.concatenate(all_dists[mt]) if all_dists[mt] else np.array([])
                violin_data[mt].append((xi, arr))
            if any(len(all_dists[mt]) for mt in METRICS):
                any_valid = True

        if not any_valid:
            continue

        fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
        fig.suptitle(
            f'Distance to full-training-background reference — {DATASET_LABELS[ds]} / {MODEL_LABELS[model]}',
            fontsize=12, fontweight='bold'
        )

        for ax, mt in zip(axes, METRICS):
            nonempty = [(xi, arr) for xi, arr in violin_data[mt] if len(arr)]
            if nonempty:
                styled_violin(ax,
                              [arr for _, arr in nonempty],
                              [xi  for xi, _  in nonempty],
                              [METHOD_COLORS[METHODS[xi]] for xi, _ in nonempty], metric=mt)

            bl_patch = add_baseline_true(ax, mt, ds)
            ax.set_xticks(range(len(METHODS)))
            ax.set_xticklabels([METHOD_LABELS[m] for m in METHODS], rotation=15, fontsize=13)
            ax.set_xlim(-0.6, len(METHODS) - 0.4)
            ax.set_ylabel('Reference comparisons (%)' if mt == 'jac' else METRIC_YLABEL[mt], fontsize=14)
            ax.set_xlabel('Sampling Method', fontsize=14)
            ax.tick_params(axis='x', length=0, labelsize=13)
            ax.tick_params(axis='y', labelsize=13)

        method_patches = [mpatches.Patch(facecolor=METHOD_COLORS[m], alpha=0.75,
                                          label=METHOD_LABELS[m]) for m in METHODS]
        handles = method_patches + ([bl_patch] if bl_patch else [])
        axes[0].legend(handles=handles, fontsize=9, loc='upper right')

        fig.tight_layout()
        fname = os.path.join(FIG_TRUE_DIR, f'true_violin_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')